# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [5]:
print("Files in the data folder:", sorted(p.name for p in (ROOT / "data").glob("*")))
print("Locations of penguins_raw.csv:", list(ROOT.rglob("penguins_raw.csv")))

Files in the data folder: ['countries.csv', 'penguins_raw.csv', 'species_lookup.csv', 'urban_population.csv']
Locations of penguins_raw.csv: [PosixPath('/workspaces/data-infra-project/data/penguins_raw.csv')]


In [6]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()


(344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [7]:
raw.isna().sum()

studyName                0
Sample Number            0
Species                  0
Region                   0
Island                   0
Stage                    0
Individual ID            0
Clutch Completion        0
Date Egg                 0
Culmen Length (mm)       2
Culmen Depth (mm)        2
Flipper Length (mm)      2
Body Mass (g)            2
Sex                     11
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Comments               290
dtype: int64

Look at the species column. What would make it hard to use as a key?

In [8]:
raw["Species"].value_counts()

Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [9]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

['study',
 'Sample Number',
 'species_raw',
 'Region',
 'island',
 'Stage',
 'individual_id',
 'clutch_completion',
 'date_egg',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex',
 'Delta 15 N (o/oo)',
 'Delta 13 C (o/oo)',
 'Comments']

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [10]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

,species_raw,latin_name,sex
0,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,male
1,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,female
3,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,NaN
152,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,female
153,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,male
178,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,NaN


**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [11]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

study                           str
individual_id                   str
latin_name                      str
island                          str
date_egg             datetime64[us]
clutch_completion               str
bill_length_mm              float64
bill_depth_mm               float64
flipper_length_mm           float64
body_mass_g                 float64
sex                             str
dtype: object

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [12]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

rows in pandas: 344


rows_in_duckdbBIGINT
344


In [13]:
con.sql("DESCRIBE penguins_clean")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
study,VARCHAR,YES,NULL,NULL,NULL
individual_id,VARCHAR,YES,NULL,NULL,NULL
latin_name,VARCHAR,YES,NULL,NULL,NULL
island,VARCHAR,YES,NULL,NULL,NULL
date_egg,TIMESTAMP,YES,NULL,NULL,NULL
clutch_completion,VARCHAR,YES,NULL,NULL,NULL
bill_length_mm,DOUBLE,YES,NULL,NULL,NULL
bill_depth_mm,DOUBLE,YES,NULL,NULL,NULL
flipper_length_mm,DOUBLE,YES,NULL,NULL,NULL
body_mass_g,DOUBLE,YES,NULL,NULL,NULL


## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [14]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

rowsBIGINT,distinct_idsBIGINT
344,190


In [15]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

studyVARCHAR,latin_nameVARCHAR,islandVARCHAR,individual_idVARCHAR
PAL0708,Pygoscelis adeliae,Biscoe,N11A1
PAL0809,Pygoscelis papua,Biscoe,N11A1


**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [16]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

common_nameVARCHAR,latin_nameVARCHAR,genusVARCHAR,group_nameVARCHAR
Adélie penguin,Pygoscelis adeliae,Pygoscelis,brush-tailed penguins
Chinstrap penguin,Pygoscelis antarcticus,Pygoscelis,brush-tailed penguins
Gentoo penguin,Pygoscelis papua,Pygoscelis,brush-tailed penguins
Emperor penguin,Aptenodytes forsteri,Aptenodytes,great penguins


Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [18]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

latin_nameVARCHAR,nBIGINT


## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [19]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

rows_after_joinBIGINT
276


**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [20]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

latin_nameVARCHAR,unmatched_rowsBIGINT
Pygoscelis antarctica,68


Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [21]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

rows_after_joinBIGINT
344


Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [22]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

common_nameVARCHAR,penguin_rowsBIGINT
Adélie penguin,152
Gentoo penguin,124
Chinstrap penguin,68
Emperor penguin,0


## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [23]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

rows_after_joinBIGINT
468


**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [24]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

common_nameVARCHAR,islandVARCHAR,penguinsBIGINT,avg_mass_gDOUBLE
Adélie penguin,Biscoe,44,3710.0
Adélie penguin,Dream,56,3688.0
Adélie penguin,Torgersen,52,3706.0
Chinstrap penguin,Dream,68,3733.0
Gentoo penguin,Biscoe,124,5076.0


### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: your own two tables

Apply the checklist to the two tables you chose in Lab 1. If you'd like to swap datasets, that's fine; you just need two public tables that share a column.

## B1. Load both tables with pandas

How does the median share of people living in urban areas differ across world regions in the most recent year with broad country coverage? I’m interested in how urbanization varies across regions because it provides context for comparing urban development patterns.

In [25]:
FILE_A = "urban_population.csv"
FILE_B = "countries.csv"

a = pd.read_csv(ROOT / "data" / FILE_A)
b = pd.read_csv(ROOT / "data" / FILE_B)
print("A:", a.shape, "  B:", b.shape)
a.head()

A: (6650, 4)   B: (265, 5)


,country_name,country_code,year,urban_pct
0,Aruba,ABW,2000,65.354550
1,Africa Eastern and Southern,AFE,2000,28.468688
2,Afghanistan,AFG,2000,18.558200
3,Africa Western and Central,AFW,2000,38.859933
4,Angola,AGO,2000,50.507065


In [26]:
b.head()

,Country Code,Region,IncomeGroup,SpecialNotes,TableName
0,ABW,Latin America & Caribbean,High income,NaN,Aruba
1,AFE,NaN,NaN,"26 countries, stretching from the Red Sea in t...",Africa Eastern and Southern
2,AFG,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan
3,AFW,NaN,NaN,"22 countries, stretching from the westernmost ...",Africa Western and Central
4,AGO,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola


**Grain** (double-click to edit). One row in table A is: _the urban population share for one country, economy, or aggregate area in one year__. One row in table B is: _one country, economy, or aggregate area and contains metadata such as its region and income group__.

## B2. Clean: at least three steps

Use the Part A steps as a model: rename columns, standardize text, fix types, drop columns you don't need. Write each step in its own line with a comment saying *why*.

In [29]:
# Check data quality before cleaning.
b_key = "Country Code" if "Country Code" in b.columns else "country_code"

print("Rows in A and B:", len(a), len(b))
print("A missing values:\n", a[["country_code", "year", "urban_pct"]].isna().sum())
print("B missing values:\n", b[[b_key, "Region"]].isna().sum())
print("A field types:\n", a[["country_code", "year", "urban_pct"]].dtypes)

urban_numeric = pd.to_numeric(a["urban_pct"], errors="coerce")
print("A non-numeric urban shares:", (a["urban_pct"].notna() & urban_numeric.isna()).sum())
print("A urban shares outside 0–100:", ((urban_numeric < 0) | (urban_numeric > 100)).sum())

# Step 1: Rename the lookup key to match the main table.
b = b.rename(columns={b_key: "country_code"})

# Step 2: Check whether the keys need text standardization.
for label, table in [("A", a), ("B", b)]:
    codes = table["country_code"].astype("string")
    standardized = codes.str.strip().str.upper()
    print(f"{label} codes needing standardization:", codes.ne(standardized).sum())
    print(f"{label} key examples:", codes.dropna().unique()[:5].tolist())

# Step 3: Keep the existing types and retain all rows for the join checks.
print("A duplicate code-year rows:", a.duplicated(["country_code", "year"]).sum())
print("B duplicate code rows:", b["country_code"].duplicated().sum())

a.head()

Rows in A and B: 6650 265
A missing values:
 country_code     0
year             0
urban_pct       25
dtype: int64
B missing values:
 country_code     0
Region          48
dtype: int64
A field types:
 country_code     string
year              int64
urban_pct       float64
dtype: object
A non-numeric urban shares: 0
A urban shares outside 0–100: 0
A codes needing standardization: 0
A key examples: ['ABW', 'AFE', 'AFG', 'AFW', 'AGO']
B codes needing standardization: 0
B key examples: ['ABW', 'AFE', 'AFG', 'AFW', 'AGO']
A duplicate code-year rows: 0
B duplicate code rows: 0


,country_name,country_code,year,urban_pct
0,Aruba,ABW,2000,65.354550
1,Africa Eastern and Southern,AFE,2000,28.468688
2,Afghanistan,AFG,2000,18.558200
3,Africa Western and Central,AFW,2000,38.859933
4,Angola,AGO,2000,50.507065


**Data quality and cleaning.** Table A has 6,650 rows and Table B has 265 rows. Neither table has missing country codes. Table A has 25 missing urban population shares, and all recorded shares are numeric and fall within 0–100%. Table B has 48 missing region labels. I standardized the country codes in both tables by trimming whitespace and converting them to uppercase. No codes changed. Before and after cleaning, Table A had no duplicate country-code–year pairs, and Table B had no duplicate country codes. I retained rows with missing shares or region labels for the join checks.

## B3. Load into DuckDB and check the key

In [41]:
import duckdb

con = duckdb.connect()

# Load the two DataFrames prepared in B2.
con.execute("CREATE OR REPLACE TABLE table_a AS SELECT * FROM a")
con.execute("CREATE OR REPLACE TABLE table_b AS SELECT * FROM b")

KEY_A = "country_code"
KEY_B = "country_code"

# Check for repeated country codes in the lookup table.
display(con.sql(f"""
    SELECT "{KEY_B}", COUNT(*) AS n
    FROM table_b
    GROUP BY "{KEY_B}"
    HAVING COUNT(*) > 1
    ORDER BY "{KEY_B}"
    LIMIT 10
""").df())


,country_code,n


The lookup table has no duplicate country codes, so each population record can match at most one row in the lookup table.

## B4. Join and count

In [42]:
display(con.sql("""
    SELECT COUNT(*) AS rows_in_table_a
    FROM table_a
""").df())

display(con.sql(f"""
    SELECT COUNT(*) AS rows_after_join
    FROM table_a a
    JOIN table_b b
      ON a."{KEY_A}" = b."{KEY_B}"
""").df())

,rows_in_table_a
0,6650


,rows_after_join
0,6625


## B5. Find what didn't match

In [43]:
con.sql(f"""
    SELECT a."{KEY_A}", count(*) AS unmatched_rows
    FROM table_a a
    LEFT JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b."{KEY_B}" IS NULL
    GROUP BY a."{KEY_A}"
    ORDER BY unmatched_rows DESC
    LIMIT 20
""")

country_codeVARCHAR,unmatched_rowsBIGINT
INX,25


**What didn't match, and what will you do about it?** (fix the key, drop those rows, or keep them as unknown, and why)



All 25 unmatched rows had code INX, which was missing from the lookup table. I kept them in the source data but excluded them from the regional comparison because they had no region assignment.

## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [45]:
con.sql("""
    WITH eligible AS (
        SELECT a.year, a.urban_pct, b.Region AS region
        FROM table_a a
        JOIN table_b b ON a.country_code = b.country_code
        WHERE a.urban_pct IS NOT NULL
          AND NULLIF(TRIM(b.Region), '') IS NOT NULL
    ),
    selected_year AS (
        SELECT year
        FROM eligible
        GROUP BY year
        ORDER BY COUNT(*) DESC, year DESC
        LIMIT 1
    )
    SELECT year, region,
           COUNT(*) AS economies,
           ROUND(MEDIAN(urban_pct), 2) AS median_urban_pct
    FROM eligible
    WHERE year = (SELECT year FROM selected_year)
    GROUP BY year, region
    ORDER BY median_urban_pct DESC
""")

yearBIGINT,regionVARCHAR,economiesBIGINT,median_urban_pctDOUBLE
2024,North America,3,82.7
2024,Middle East & North Africa,23,79.44
2024,Latin America & Caribbean,42,71.82
2024,Europe & Central Asia,58,69.15
2024,East Asia & Pacific,37,64.73
2024,Sub-Saharan Africa,48,44.43
2024,South Asia,6,38.69


**What does the result tell you?** (one or two sentences)

In 2024, North America had the highest median urban population share (82.70%), while South Asia had the lowest (38.69%). North America’s median was based on three economies.
## B7. Submit

1. Run the cell below.
2. Answer the two reflection questions in `labs/lab2.md`.
3. Save this notebook (Cmd+S), then commit and push: **Source Control** → `Lab 2` → **Commit** → **Sync Changes**.
4. Paste your repo URL into the Lab 2 assignment on CourseWorks.

In [ ]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")